Method inspired by "Self-Routing" (arxiv:2604:00421) Mohamud et.al.

- Language model already have "inner-monologue", those are its hidden representations 

- Instead of "training LLM to create its inner-monologue", we "interpret LLM's inner-monologue"

Given a hidden representation $h_{1:t} = \phi(x_{1:t})$, and a abstract vocabulary of size V, we slice $(h_{t}^{D-V+1}, h_{t}^{D-V+2}, \dots, h_{t}^{D})$ as the abstract logits

- This builds on the assumption that "data-dependent routing" do NOT need to be plastic (Kenyon layer hypothesis)

#### Implementation

- For SoRLWrapper, We can set lm_head.weight[-V_abs:][-V_abs:] to diagonal matrix, and we freeze this lm_head.weight[-V_abs:], thereby making the abstraction choice deterministic

- There are train-test mismatch, due to Jacobi decoding iterations < num of abstract tokens in the sequence, but we just want to try it out

- We need only traj_loss and nothing else, abstraction embedding remains trainable


In [1]:
# An equivalent way to implement this, is to set abs_
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

# Disable MPS for stability
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False

from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper

from sorl.trainer_ablate import SoRLTrainerv2, SoRLTrainerv3
from sorl.trainer_ablate import SoRLConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Using device: cpu


In [ ]:
# Initialize SoRL model
from sorl.sorl_wrapper import SorlModelWrapper
model_name = "Qwen/Qwen3-0.6B"
model = SorlModelWrapper.from_pretrained(
    model_name,
    abstract_vocab_size_list=[128],
)
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)



Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [51]:
# (1). Initialize the lm_head, and add gradient hook
import torch

nl_vocab = model.vocab_sizes[0]
abs_vocab = model.vocab_sizes[1]

# (1). Initialize abstraction projection parameters
abs_proj = torch.diag(torch.cat([torch.tensor([0.]), torch.ones(abs_vocab-1)]), diagonal=0)
model.model.lm_head.weight.data[nl_vocab:] = 0
model.model.lm_head.weight.data[nl_vocab:, -abs_vocab:] = abs_proj
model.model.lm_head.weight.data[nl_vocab:]

# (2). Register backward hook to freeze lm_head abstract rows ─────────
def _freeze_abs_lm_head_hook(grad):
    grad = grad.clone()
    grad[nl_vocab:, :] = 0.0
    return grad

_hook_handle = model.model.lm_head.weight.register_hook(_freeze_abs_lm_head_hook)
print(f"Registered backward hook to freeze lm_head rows [{nl_vocab}:]")


# (3). Sanity Check
with torch.no_grad():
    dummy_ids = tokenizer("Hello world", return_tensors="pt")["input_ids"].to(device)
    h = model.model.model(dummy_ids).last_hidden_state[0, -1]
    logits = model.model(dummy_ids).logits[0, -1]
    match = torch.allclose(logits[nl_vocab+1:], h[-abs_vocab+1:], atol=1e-4)
    print(f"Sanity: abs_logits == h[-{abs_vocab}:]? {match}")
    print("Remark: We ignore <mask_abs> for now, if this mechanism work, we need to retire <mask_abs>")

Registered backward hook to freeze lm_head rows [151936:]
Sanity: abs_logits == h[-129:]? True
Remark: We ignore <mask_abs> for now, if this mechanism work, we need to retire <mask_abs>


In [ ]:
# ── Helper functions for self-routing training ──────────────────────────
from sorl.sorl_trainer import infer_insert_mask, expand_prompt_len, insert_tokens_with_padding
import torch.nn as nn

def insert_and_recursion(model, input_ids, attention_mask, prompt_len, pad_token_id,
                         K=4, max_iterations=2, memory_span_abs=1024, memory_span_traj=1024,
                         temperature=0.0, response_only_abs=False):
    """Insert abstract placeholders, run Jacobi recursion, return expanded sequence + logits."""
    insert_mask = infer_insert_mask(input_ids, K, attention_mask,
                                    prompt_len=prompt_len if response_only_abs else None)
    exp_pl = expand_prompt_len(prompt_len, insert_mask)
    exp_data, exp_attn = insert_tokens_with_padding(
        input_ids, attention_mask, insert_mask, model.vocab_sizes[0], pad_token_id)
    
    data, ppt, logits = model.recursion(
        exp_data, exp_attn, max_iterations=max_iterations,
        memory_span_abs=memory_span_abs, memory_span_traj=memory_span_traj,
        temperature=temperature, prompt_len=exp_pl)
        
    return data, exp_attn, exp_pl, logits

def traj_loss_from_logits(logits, data, attn_mask, prompt_len, base_vocab):
    """Compute traj_loss from logits: mask abstract logits, CE on base-vocab response positions."""
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = data[..., 1:].contiguous()
    shift_attn = attn_mask[..., 1:].clone()

    # mask prompt
    sidx = torch.arange(shift_attn.size(1), device=data.device).unsqueeze(0)
    shift_attn[sidx < (prompt_len.unsqueeze(1) - 1)] = 0

    # traj mask: base-vocab target positions only
    traj_mask = ((data[:, 1:] < base_vocab).float()) * shift_attn.float()

    # mask abstract logits → clean softmax over base vocab
    traj_logits = shift_logits.clone()
    traj_logits[..., base_vocab:] = -float("inf")
    safe_labels = shift_labels.clone()
    safe_labels[~traj_mask.bool()] = 0

    loss_fct = nn.CrossEntropyLoss(reduction="none")
    per_tok = loss_fct(traj_logits.view(-1, traj_logits.size(-1)), safe_labels.view(-1))
    return (per_tok.view(data.shape[0], -1) * traj_mask).sum() / traj_mask.sum().clamp(min=1)

print("Helpers loaded: insert_and_recursion, traj_loss_from_logits")

Helpers loaded: insert_and_recursion, traj_loss_from_logits


In [ ]:
MEM_ABS, MEM_TRAJ = 1024, 1024
MAX_ITERS = 2

# (3). Insert abstract placeholders + Jacobi recursion (no grad, policy is fixed)
with torch.no_grad():
    data, exp_attn, exp_pl, logits = insert_and_recursion(
        model, input_ids, attention_mask, prompt_len, pad_token_id,
        K=K, max_iterations=MAX_ITERS, memory_span_abs=MEM_ABS, memory_span_traj=MEM_TRAJ,
        temperature=0.0)

# 2. Traj loss from logits (no extra forward pass)
traj_loss = traj_loss_from_logits(logits, data, exp_attn, exp_pl, nl_vocab)

print(f"data shape: {data.shape}, traj_loss: {traj_loss.item():.4f}")

data shape: torch.Size([2, 313]), traj_loss: 1.5899


In [83]:
# ── (4a). Training Config & Setup ───────────────────────────────────────
from data.pt_dataset import GSM8KDataset, collate_fn, evaluate_accuracy
from sorl.trainer_ablate import _get_lr
from torch.utils.data import DataLoader

K = 4; MAX_ITERS = 2; MEM_ABS = 1024; MEM_TRAJ = 1024; TEMPERATURE = 0.0
LR = 1e-5; EMB_LR_MULT = 1.0; WARMUP_STEPS = 50; COOLDOWN_FRAC = 0.4
MAX_GRAD_NORM = 1.0; BATCH_SIZE = 2; GRAD_ACCUM = 1; NUM_EPOCHS = 3
LOG_EVERY = 10; EVAL_EVERY = 500; EVAL_SAMPLES = 100; EVAL_K = 4
SAVE_DIR = "./ckpt/self_routing"

train_ds = GSM8KDataset(split="train", tokenizer=tokenizer, max_length=256)
val_ds   = GSM8KDataset(split="test",  tokenizer=tokenizer, max_length=256)
dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
total_steps = len(dl) * NUM_EPOCHS // GRAD_ACCUM

base_vocab_int = int(nl_vocab.item())
pad_token_id = tokenizer.pad_token_id

emb_params, other_params = [], []
for name, p in model.named_parameters():
    (emb_params if ("embed_tokens" in name or "lm_head" in name) else other_params).append(p)
optimizer = torch.optim.AdamW([
    {"params": other_params, "lr": LR},
    {"params": emb_params,   "lr": LR * EMB_LR_MULT},
], weight_decay=0.01)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, steps/epoch: {len(dl)}, total: {total_steps}")

Train: 7473, Val: 1319, steps/epoch: 3737, total: 11211


In [ ]:
# ── (4b). Training Loop ─────────────────────────────────────────────────
os.makedirs(SAVE_DIR, exist_ok=True)
history = {"step": [], "traj_loss": [], "base_loss": [], "lr": []}
model.train()
global_step = 0
t0 = time.time()

for epoch in range(NUM_EPOCHS):
    for batch_idx, batch in enumerate(dl):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        prompt_len = batch["prompt_len"].to(device)

        # LR schedule
        lr = _get_lr(global_step, total_steps, WARMUP_STEPS, COOLDOWN_FRAC, LR)
        optimizer.param_groups[0]["lr"] = lr
        optimizer.param_groups[1]["lr"] = lr * EMB_LR_MULT

        # Base traj loss (logging only)
        with torch.no_grad():
            labels = input_ids.clone()
            labels[attention_mask == 0] = -100
            si = torch.arange(labels.size(1), device=device).unsqueeze(0)
            labels[si < prompt_len.unsqueeze(1)] = -100
            out = model(input_ids=input_ids, attention_mask=attention_mask,
                        memory_span_abs=MEM_ABS, memory_span_traj=MEM_TRAJ)
            lg = out.logits.clone(); lg[:, :, base_vocab_int:] = -float("inf")
            base_loss = nn.CrossEntropyLoss(ignore_index=-100)(
                lg[:, :-1].contiguous().view(-1, lg.size(-1)),
                labels[:, 1:].contiguous().view(-1))
            del out, lg

        # Insert + recursion (no grad, policy is fixed)
        # - Wrong, recursion produces logits without gradient, whilst we need it to contain gradient
        
        data, exp_attn, exp_pl, logits = insert_and_recursion(
            model, input_ids, attention_mask, prompt_len, pad_token_id,
            K=K, max_iterations=MAX_ITERS,
            memory_span_abs=MEM_ABS, memory_span_traj=MEM_TRAJ,
            temperature=TEMPERATURE)

        # Traj loss (with grad)
        traj_loss = traj_loss_from_logits(logits, data, exp_attn, exp_pl, base_vocab_int)
        (traj_loss / GRAD_ACCUM).backward()

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            if MAX_GRAD_NORM > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        # Logging
        if (batch_idx + 1) % LOG_EVERY == 0:
            elapsed = time.time() - t0
            frac = max(global_step, 1) / max(total_steps, 1)
            eta = elapsed / frac * (1 - frac) if frac > 0 else 0
            em, es = divmod(int(eta), 60); eh, em = divmod(em, 60)
            print(f"[{epoch+1}/{NUM_EPOCHS}] step {global_step}/{total_steps} "
                  f"traj={traj_loss.item():.4f} base={base_loss.item():.4f} "
                  f"lr={lr:.2e} ETA {eh}h{em:02d}m{es:02d}s")
            history["step"].append(global_step)
            history["traj_loss"].append(traj_loss.item())
            history["base_loss"].append(base_loss.item())
            history["lr"].append(lr)

        del traj_loss, logits
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        # Eval
        if global_step > 0 and global_step % EVAL_EVERY == 0:
            res = evaluate_accuracy(model, tokenizer, val_ds, device,
                                    num_samples=EVAL_SAMPLES, eval_K=EVAL_K)
            print(f"  === Eval step {global_step}: {res} ===")

    print(f"=== Epoch {epoch+1} complete ===")

# Final eval
res = evaluate_accuracy(model, tokenizer, val_ds, device,
                        num_samples=EVAL_SAMPLES, eval_K=EVAL_K)
print(f"Final eval: {res}")
print(f"Done. {global_step} steps in {time.time()-t0:.0f}s")